# VAE + NN Regression
This notebook aims to perform a regression on the production of solar, wind and hydro renewable energy. To achieve this, we are going to use a VAE + NN configuration. The VAE is going to be in charge of reducing the dimensionality of the problem, while the NN will perform the regression itself.

## 0: Importing libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
# from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

import optuna
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import json
import os

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import TimeSeriesSplit

from datetime import date

## 1: Import the data
We want to use as features the meteo forecast data that are outputted from notebook 1, while as label the outputs of notebook 2. Is important to **select zone and energy type** here.

In [2]:
# LUCA's IMPORTS
sel_zone = 'IT_SICI'
energies = ['solar','wind','hydro']
detrended_energies = ['detrended_solar','detrended_wind','detrended_hydro']
energy_colors = ['red','green','blue']
target_index = 1

df_features = pd.read_csv(f'1_output/{sel_zone}_meteo_forecast_data.csv')
df_labels = pd.read_csv(f'2_outputs/{sel_zone}_installation_detrended_actuals.csv')

# remove timezone information
df_labels['date_time'] = pd.to_datetime(df_labels['date_time'], utc= True).dt.tz_convert(None)
# sort by date
df_labels = df_labels.sort_values('date_time')
df_labels = df_labels.set_index('date_time')
# resample every 3h with no overlapping windows and keeping the first timestamp as index
start_time = df_labels.index[0]
groups = []
# Define which columns to sum and which to average
sum_columns = ['solar','wind','hydro','load','detrended_solar','detrended_wind','detrended_hydro']
mean_columns = ['ratio','solar_installed','wind_installed','hydro_installed']  # example

groups = []
start_time = df_labels.index[0]

while start_time + pd.Timedelta(hours=3) <= df_labels.index[-1] + pd.Timedelta(minutes=1):
    end_time = start_time + pd.Timedelta(hours=3)
    chunk = df_labels[start_time:end_time - pd.Timedelta(seconds=1)]

    if len(chunk) == 3 and all((chunk.index == [start_time + pd.Timedelta(hours=i) for i in range(3)])):
        summed = chunk[sum_columns].sum()
        averaged = chunk[mean_columns].mean()
        
        combined = pd.concat([summed, averaged])
        combined.name = start_time
        groups.append(combined)

    start_time += pd.Timedelta(hours=3)

# retrieve date time info
result_df = pd.DataFrame(groups).reset_index().rename(columns={'index': 'date_time'})

df_labels = result_df

# normalize time to range [0, 1]
min_time = df_labels['date_time'].min()
max_time = df_labels['date_time'].max()
df_labels['time'] = (df_labels['date_time'] - min_time) / (max_time - min_time)
df_labels['time'] = df_labels['time'].astype(float)  # convert timedelta to float

# keep only desired columns
df_labels = df_labels[['date_time', 'time', 'detrended_solar', 'detrended_wind', 'detrended_hydro']]
df_labels 

,date_time,time,detrended_solar,detrended_wind,detrended_hydro
0,2022-01-01 00:00:00,0.000000,0.000000,0.493065,0.664474
1,2022-01-01 03:00:00,0.000114,0.000000,0.552320,0.664474
2,2022-01-01 06:00:00,0.000228,0.410682,0.767952,0.644737
3,2022-01-01 09:00:00,0.000343,1.019868,0.567734,0.638158
4,2022-01-01 12:00:00,0.000457,0.766827,0.735547,0.644737
...,...,...,...,...,...
8744,2024-12-30 06:00:00,0.999543,0.435416,0.006167,0.177632
8745,2024-12-30 09:00:00,0.999657,1.100819,0.006167,0.177632
8746,2024-12-30 12:00:00,0.999772,0.772046,0.006989,0.177632
8747,2024-12-30 15:00:00,0.999886,0.023153,0.012334,0.177632


In [3]:
df_features

,Unnamed: 0,year_t-8,2t_t-8,solar_t-8,tp_t-8,ws_10m_t-8,ws_100m_t-8,hour_sin_t-8,hour_cos_t-8,day_sin_t-8,...,ws_100m_t-0,hour_sin_t-0,hour_cos_t-0,day_sin_t-0,day_cos_t-0,month_sin_t-0,month_cos_t-0,sin_dayofyear_t-0,cos_dayofyear_t-0,timestamp
0,0,-1.225088,-1.266874,-0.840241,-0.346102,-0.499167,-0.136226,0.271583,2.018211,0.271472,...,0.155427,-0.658808,1.520038,0.271765,1.404013,0.703139,1.138456,0.018745,1.314406,2022-01-01 21:00:00
1,1,-1.225088,-1.061319,-0.840241,-0.346102,-0.458560,-0.232371,1.202105,1.519808,0.271472,...,-0.073153,0.271693,2.018495,0.543513,1.317933,0.703139,1.138456,0.043657,1.313790,2022-01-02 00:00:00
2,2,-1.225088,-1.108253,-0.552594,-0.335803,-0.396763,-0.119983,1.587539,0.316557,0.271472,...,-0.172147,1.202194,1.520038,0.543513,1.317933,0.703139,1.138456,0.043657,1.313790,2022-01-02 03:00:00
3,3,-1.225088,-0.506775,0.606941,-0.335803,-0.356149,-0.240595,1.202105,-0.886694,0.271472,...,-0.480886,1.587620,0.316654,0.543513,1.317933,0.703139,1.138456,0.043657,1.313790,2022-01-02 06:00:00
4,4,-1.225088,-0.134538,0.650498,-0.335803,0.090780,-0.251705,0.271583,-1.385097,0.271472,...,-0.891937,1.202194,-0.886729,0.543513,1.317933,0.703139,1.138456,0.043657,1.313790,2022-01-02 09:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8017,8017,1.224173,-0.489254,-0.505522,-0.347672,-1.218649,-1.093554,-0.658939,-0.886694,-0.295205,...,-1.288107,0.271693,-1.385187,-0.011584,1.433103,-0.018562,1.324666,-0.006173,1.314612,2024-12-31 12:00:00
8018,8018,1.224173,-1.086732,-0.840241,-0.348381,-1.314590,-1.271872,-1.044373,0.316557,-0.295205,...,-0.856248,-0.658808,-0.886729,-0.011584,1.433103,-0.018562,1.324666,-0.006173,1.314612,2024-12-31 15:00:00
8019,8019,1.224173,-1.086732,-0.840241,-0.348381,-1.314590,-1.271872,-1.044373,0.316557,-0.295205,...,-0.966149,-1.044234,0.316654,-0.011584,1.433103,-0.018562,1.324666,-0.006173,1.314612,2024-12-31 18:00:00
8020,8020,1.224173,-1.397535,-0.840241,-0.348381,-1.438111,-1.504216,-0.658939,1.519808,-0.295205,...,-0.966149,-1.044234,0.316654,-0.011584,1.433103,-0.018562,1.324666,-0.006173,1.314612,2024-12-31 18:00:00


### 1.1: Remove duplicated time information
Here we remove the duplicated time informations, such as day and month (we already have dayoftheyear that represents this same information)

In [4]:
# we use as temporal info the day of the year (1...365)
features = ['day_sin_t-0','day_cos_t-0',
           'day_sin_t-1','day_cos_t-1',
           'day_sin_t-2','day_cos_t-2',
           'day_sin_t-3','day_cos_t-3',
           'day_sin_t-4','day_cos_t-4',
           'day_sin_t-5','day_cos_t-5',
           'day_sin_t-6','day_cos_t-6',
           'day_sin_t-7','day_cos_t-7',
           'day_sin_t-8','day_cos_t-8',
           'month_sin_t-0','month_cos_t-0',
           'month_sin_t-1','month_cos_t-1',
           'month_sin_t-2','month_cos_t-2',
           'month_sin_t-3','month_cos_t-3',
           'month_sin_t-4','month_cos_t-4',
           'month_sin_t-5','month_cos_t-5',
           'month_sin_t-6','month_cos_t-6',
           'month_sin_t-7','month_cos_t-7',
           'month_sin_t-8','month_cos_t-8']

# in alternative one can use the following lines to keep day and month time info
'''
features = ['sin_dayofyear_t0','cos_dayofyear_t0',
            'sin_dayofyear_t1','cos_dayofyear_t1',
            'sin_dayofyear_t2','cos_dayofyear_t2',
            'sin_dayofyear_t3','cos_dayofyear_t3',
            'sin_dayofyear_t4','cos_dayofyear_t4',
            'sin_dayofyear_t5','cos_dayofyear_t5',
            'sin_dayofyear_t6','cos_dayofyear_t6',
            'sin_dayofyear_t7','cos_dayofyear_t7',
            'sin_dayofyear_t8','cos_dayofyear_t8','Unnamed: 0'] '''

df_features = df_features.drop(columns = features)

In [5]:
sel_cols = df_features.columns[1:]
df_features = df_features[sel_cols]
df_features
# methereological data every 3 hours, running over the course of a day  (10 features every 3 hour = 90 features)

,year_t-8,2t_t-8,solar_t-8,tp_t-8,ws_10m_t-8,ws_100m_t-8,hour_sin_t-8,hour_cos_t-8,sin_dayofyear_t-8,cos_dayofyear_t-8,...,2t_t-0,solar_t-0,tp_t-0,ws_10m_t-0,ws_100m_t-0,hour_sin_t-0,hour_cos_t-0,sin_dayofyear_t-0,cos_dayofyear_t-0,timestamp
0,-1.225088,-1.266874,-0.840241,-0.346102,-0.499167,-0.136226,0.271583,2.018211,0.018720,1.314407,...,-0.998212,-0.840262,-0.341855,-0.189767,0.155427,-0.658808,1.520038,0.018745,1.314406,2022-01-01 21:00:00
1,-1.225088,-1.061319,-0.840241,-0.346102,-0.458560,-0.232371,1.202105,1.519808,0.018720,1.314407,...,-1.107250,-0.840262,-0.341855,-0.334269,-0.073153,0.271693,2.018495,0.043657,1.313790,2022-01-02 00:00:00
2,-1.225088,-1.108253,-0.552594,-0.335803,-0.396763,-0.119983,1.587539,0.316557,0.018720,1.314407,...,-1.078870,-0.840262,-0.341855,-0.335668,-0.172147,1.202194,1.520038,0.043657,1.313790,2022-01-02 03:00:00
3,-1.225088,-0.506775,0.606941,-0.335803,-0.356149,-0.240595,1.202105,-0.886694,0.018720,1.314407,...,-1.183218,-0.528330,-0.345909,-0.508541,-0.480886,1.587620,0.316654,0.043657,1.313790,2022-01-02 06:00:00
4,-1.225088,-0.134538,0.650498,-0.335803,0.090780,-0.251705,0.271583,-1.385097,0.018720,1.314407,...,-0.446864,0.732986,-0.345909,-0.900305,-0.891937,1.202194,-0.886729,0.043657,1.313790,2022-01-02 09:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8017,1.224173,-0.489254,-0.505522,-0.347672,-1.218649,-1.093554,-0.658939,-0.886694,-0.031049,1.314408,...,-0.474199,0.765375,-0.346693,-1.213290,-1.288107,0.271693,-1.385187,-0.006173,1.314612,2024-12-31 12:00:00
8018,1.224173,-1.086732,-0.840241,-0.348381,-1.314590,-1.271872,-1.044373,0.316557,-0.031049,1.314408,...,-0.668105,-0.481063,-0.346693,-0.904086,-0.856248,-0.658808,-0.886729,-0.006173,1.314612,2024-12-31 15:00:00
8019,1.224173,-1.086732,-0.840241,-0.348381,-1.314590,-1.271872,-1.044373,0.316557,-0.031049,1.314408,...,-1.289500,-0.840262,-0.345005,-0.969932,-0.966149,-1.044234,0.316654,-0.006173,1.314612,2024-12-31 18:00:00
8020,1.224173,-1.397535,-0.840241,-0.348381,-1.438111,-1.504216,-0.658939,1.519808,-0.031049,1.314408,...,-1.289500,-0.840262,-0.345005,-0.969932,-0.966149,-1.044234,0.316654,-0.006173,1.314612,2024-12-31 18:00:00


In [6]:
df_labels

,date_time,time,detrended_solar,detrended_wind,detrended_hydro
0,2022-01-01 00:00:00,0.000000,0.000000,0.493065,0.664474
1,2022-01-01 03:00:00,0.000114,0.000000,0.552320,0.664474
2,2022-01-01 06:00:00,0.000228,0.410682,0.767952,0.644737
3,2022-01-01 09:00:00,0.000343,1.019868,0.567734,0.638158
4,2022-01-01 12:00:00,0.000457,0.766827,0.735547,0.644737
...,...,...,...,...,...
8744,2024-12-30 06:00:00,0.999543,0.435416,0.006167,0.177632
8745,2024-12-30 09:00:00,0.999657,1.100819,0.006167,0.177632
8746,2024-12-30 12:00:00,0.999772,0.772046,0.006989,0.177632
8747,2024-12-30 15:00:00,0.999886,0.023153,0.012334,0.177632


### 1.2: Joining dataframes
Since some data was discarded in previous notebooks (presence of outliers), we need to ensure that features and labels are aligned over the time information. For this reason, we perform a inner join over the dataframes, ensuring consistency over features and labels.

In [7]:
# keeping only the ones in common
df_labels['date_time'] = pd.to_datetime(df_labels['date_time']).dt.tz_localize(None)
df_features['date_time'] = pd.to_datetime(df_features['timestamp']).dt.tz_localize(None)
common = pd.merge(df_features, df_labels, on='date_time', how = 'inner')
df_labels = common[df_labels.columns]
df_features = common[df_features.columns]
print(np.shape(df_features),np.shape(df_labels))


(8002, 92) (8002, 5)


/tmp/ipykernel_271197/2463686259.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labels['date_time'] = pd.to_datetime(df_labels['date_time']).dt.tz_localize(None)


### 1.3: Setting up Training and Test data
Since our aim is to make our model being able to predict future data from past data, we are going to split training and test set according to time index: the first 80% of our data points will represent our training set, the rest test set. Computation times are already long for our implementation, so we will avoid K-Folds such as time-rollings

In [8]:
X_array = df_features.drop(columns=['timestamp','date_time']).values # 24h windows
# pick the right label
Y_array = df_labels[detrended_energies[target_index]].values
print(f'Attempting the regression on {detrended_energies[target_index]}')

n_samples = len(X_array)
split_idx = int(n_samples * 0.8)


X_train = X_array[:split_idx]
X_test = X_array[split_idx:] # we do not randomly split into training and test set to avoid replicating data across both (it's a time-series with sliding windows)

Y_train = Y_array[:split_idx]
Y_test = Y_array[split_idx:]

# keep track of the original indices and timestamps
idx_train = df_features.index[:split_idx] # to retrieve date/time at test time (for predictions)
idx_test = df_features.index[split_idx:]

test_timestamps = df_features.loc[idx_test, 'timestamp'].reset_index(drop=True)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)

# create a DataLoader for batching (training and test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Reshape the Y tensors (model consistency if we want multiple features)
Y_train_tensor = Y_train_tensor.view(-1, 1)
Y_test_tensor = Y_test_tensor.view(-1, 1)


Attempting the regression on detrended_wind


### 1.4: Make directories to save screenshots and models

In [9]:
#### make directory for saving screenshots
base_dir = "4_plots"
zone_dir = os.path.join(base_dir, sel_zone)

try:
    os.makedirs(zone_dir, exist_ok=True)
    print(f"Directory '{zone_dir}' is ready.")
except PermissionError:
    print(f"Permission denied: Unable to create '{zone_dir}'.")
except Exception as e:
    print(f"An error occurred: {e}")


#### make directory for saving screenshots
base_dir = "4_plots"
zone_dir = os.path.join(base_dir, sel_zone, energies[target_index])

try:
    os.makedirs(zone_dir, exist_ok=True)
    print(f"Directory '{zone_dir}' is ready.")
except PermissionError:
    print(f"Permission denied: Unable to create '{zone_dir}'.")
except Exception as e:
    print(f"An error occurred: {e}")

# make directory for saving weights
base_dir = "4_weights"
zone_dir = os.path.join(base_dir, sel_zone, energies[target_index])

try:
    os.makedirs(zone_dir, exist_ok=True)
    print(f"Directory '{zone_dir}' is ready.")
except PermissionError:
    print(f"Permission denied: Unable to create '{zone_dir}/{energies[target_index]}'.")
except Exception as e:
    print(f"An error occurred: {e}")

# make directory for saving studies
base_dir = "4_optuna"
zone_dir = os.path.join(base_dir, sel_zone, energies[target_index])

try:
    os.makedirs(zone_dir, exist_ok=True)
    print(f"Directory '{zone_dir}' is ready.")
except PermissionError:
    print(f"Permission denied: Unable to create '{zone_dir}/{energies[target_index]}'.")
except Exception as e:
    print(f"An error occurred: {e}")

# make directory for saving BEST MODELS
base_dir = "4_best_models"
zone_dir = os.path.join(base_dir, sel_zone, energies[target_index])

try:
    os.makedirs(zone_dir, exist_ok=True)
    print(f"Directory '{zone_dir}' is ready.")
except PermissionError:
    print(f"Permission denied: Unable to create '{zone_dir}/{energies[target_index]}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory '4_plots/IT_SICI' is ready.
Directory '4_plots/IT_SICI/wind' is ready.
Directory '4_weights/IT_SICI/wind' is ready.
Directory '4_optuna/IT_SICI/wind' is ready.
Directory '4_best_models/IT_SICI/wind' is ready.


## 2: Setup the model and Training
### 2.1: Model definition

In [10]:

class VAE(nn.Module): # VAE inherits from nn.Module, base class for neural network modules in PyTorch.
    def __init__(self, input_dim, hidden_dim1,hidden_dim2,latent_dim, hidden_dim3, activation):
        super(VAE, self).__init__() # to call init method on the parent class of VAE (nn.Module) so that is has all the same functions

        # encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim1) # hidden_dim: hidden layer neurons of encoder NN
        self.fc2 = nn.Linear(hidden_dim1,hidden_dim2)
        self.fc_mu = nn.Linear(hidden_dim2, latent_dim) # latent_dim : output layer neurons of encoder NN
        self.fc_logvar = nn.Linear(hidden_dim2, latent_dim)

        # decoder
        self.fc3 = nn.Linear(latent_dim, hidden_dim3)
        self.fc4 = nn.Linear(hidden_dim3, input_dim)
        self.activation = activation

    def encode(self, x):
        #h1 = self.activation(self.batch1(self.fc1(x)))
        h1 = self.activation((self.fc1(x)))
        h2= self.activation(self.fc2(h1))
        return self.fc_mu(h2), self.fc_logvar(h2)

    def reparameterize(self, mu, logvar): # reparametrization trick to allow backpropagation
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        #h = self.activation(self.batch2(self.fc2(z)))
        h3 = self.activation((self.fc3(z)))
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar, z  # z are the latent representations of the input x


# VAE loss function (separate function, not class method)
def vae_loss(recon_x, x, mu, logvar):
        MSE = F.mse_loss(recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return MSE + KLD, MSE, KLD



class Net(nn.Module):
    def __init__(self,Ni,N1,N2,No,activation): # Ni is the dimension of z, N1 the number of hidden neurons of the first layer, N2 the number of hidden neurons of the second layer
        super(Net,self).__init__()

        self.fc1 = nn.Linear(Ni,N1)
        self.batch1 =  nn.BatchNorm1d(N1)
        self.drop1 = nn.Dropout(p=0.1)
        
        self.fc2 = nn.Linear(N1,N2)
        self.batch2 =  nn.BatchNorm1d(N2)
        self.drop2 = nn.Dropout(p=0.1)

        # self.fc3 = nn.Linear(N2, N3)
        # self.batch3 = nn.BatchNorm1d(N3)
        # self.drop3 = nn.Dropout(p=0.1)
        
        self.out = nn.Linear(N2,No)
        self.act = activation


    def forward(self,z): # input from latent variables
            h1 = self.drop1(self.act(self.batch1(self.fc1(z))))
            h2 = self.drop2(self.act(self.batch2(self.fc2(h1))))
        
            #h1= self.act((self.fc1(z)))
            #h2 = self.act((self.fc2(h1)))
            #h3 = self.drop3(self.act(self.batch3(self.fc3(h2))))
            out = F.relu(self.out(h2))
            return (out)


    def regr_loss(self,y,output,lambda_tik):
        mse = F.mse_loss(output, y,reduction='sum')

        w_norm = 0.0
        for param in self.parameters():
            w_norm += torch.sum(torch.abs(param))

        return mse + lambda_tik*w_norm**2




# here we define the full network architecture for VAE+regression

class VAE_Regression(nn.Module):
    def __init__(self,input_dim, hidden_dim1,hidden_dim2,latent_dim,hidden_dim3,vae_activation,N1,N2,No,activation,lambda_tik):
        super(VAE_Regression,self).__init__()

        self.vae = VAE(input_dim, hidden_dim1,hidden_dim2,latent_dim, hidden_dim3, vae_activation)
        self.regression = Net(latent_dim,N1,N2,No,activation)
        
    def forward(self,x):
        reconstructed_x, mu, logvar, z = self.vae(x) # self.vae(x) is equivalent to calling self.vae.forward(x) thanks to nn.Module
        regr_output = self.regression(mu)
        return (regr_output,reconstructed_x, mu, logvar,z)

    def VAE_loss(self,reconstructed_x, x, mu, logvar): # here VAE_loss/regression_loss are methods of the VAE_Regression class to be accessed later
        return(vae_loss(reconstructed_x, x, mu, logvar))

    def regression_loss(self,y,regr_output,lambda_tik):
        return(self.regression.regr_loss(y,regr_output,lambda_tik))


### 2.2: Grid search on hyperparameters space
The aim of the following cells is to run a grid search on hyperparameters space. The only free parameter (here arbitrarily set up, according to qualitative loss trends) is $\beta$, which rescales VAE loss to be (as far as possible) comparable with regression loss. This way the machine should be able to weigh correctly the two contributes and train the VAE in the best way to perform the regression.

In [11]:
beeta = 1. / 2000
# here we perform a grid search to look for the best hyperparameters: we choose to use 4 folds. We fix as objective to minimize the average validation error over the folds
# To compute each loss estimation before udating the weights, we perform 100 evaluations of the predicted outputs and consider their average as prediction

def get_activation(name):
    return {
        "relu": nn.ReLU(),
        "sigmoid": nn.Sigmoid(),
        "leaky_relu": nn.LeakyReLU()
    }[name]

def objective(trial):

    save_dir = "4_weights_VAE_Regression/Single_hydro"
    os.makedirs(save_dir, exist_ok=True)
   
    hidden_dim1 = trial.suggest_categorical("hidden_dim1", [64,128,256]) # previous range 64-256
    hidden_dim2 = trial.suggest_categorical("hidden_dim2", [64,128,256]) #  previous range 64-256
    latent_dim = trial.suggest_categorical("latent_dim", [12,14,16,18,20]) # previous range 6-20
    hidden_dim3 = trial.suggest_categorical("hidden_dim3",[128,256] )  # previous range 64-256

    N1 = trial.suggest_categorical("N1", [64,128,256]) # previous range 16-256
    N2 = trial.suggest_categorical("N2", [64,128,256])

    lambda_tik = trial.suggest_float("lambda_tik", 5.0*1e-6,1e-4, log=True)
    lr = trial.suggest_float("lr",1e-5,5*1e-4, log=True)
    act_name = trial.suggest_categorical("activation", ["relu"]) # previous searches also with sigmoid and leaky relu 
    activation_fn = get_activation(act_name)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_epochs = 300
    batch_size = 16
    beta = beeta
    num_folds = 2
    patience= 35
    tscv = TimeSeriesSplit(n_splits=num_folds)

    epoch_train_losses = np.zeros(num_epochs)
    epoch_val_losses = np.zeros(num_epochs)
    epoch_val_vae_losses = np.zeros(num_epochs)
    epoch_val_regr_losses = np.zeros(num_epochs)
    fold_last10_regr_losses = []
    fold_model_paths = []


    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_tensor)):

        X_train_fold, Y_train_fold = X_train_tensor[train_idx], Y_train_tensor[train_idx]
        X_val_fold, Y_val_fold = X_train_tensor[val_idx], Y_train_tensor[val_idx]

        train_dataset = TensorDataset(X_train_fold, Y_train_fold)
        val_dataset = TensorDataset(X_val_fold, Y_val_fold)

        #vae_act = get_activation("relu")

        model = VAE_Regression(
            input_dim=X_train_fold.shape[1],
            # hidden_dim1=vae_best_params["H_DIM"],
            # latent_dim=vae_best_params["Z_DIM"],
            # hidden_dim2=vae_best_params["H_DIM"],
            hidden_dim1=hidden_dim1,
            hidden_dim2=hidden_dim2,
            latent_dim=latent_dim,
            hidden_dim3=hidden_dim3,
            vae_activation= activation_fn,
            N1=N1,
            N2=N2,
            No=Y_train_fold.shape[1],
            activation=activation_fn,
            lambda_tik=lambda_tik
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.25,       # wait 20 epochs of no improvement after which shrink LR by half
            patience=20)      

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        val_regr_loss_per_epoch = []
        best_val_regr = float('inf')
        early_stop_counter = 0
        best_model_state = None

        for epoch in range(num_epochs):

            model.train()
            train_loss_epoch = 0
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()

                regr_output, recon_x, mu, logvar, _ = model(x_batch)
                vae_loss = model.VAE_loss(recon_x, x_batch, mu, logvar)[0]
                regr_loss = model.regression_loss(y_batch, regr_output, lambda_tik)

                loss = (1 - beta) * regr_loss + beta * vae_loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss_epoch += loss.item()

            train_loss_epoch /= len(train_loader)
            scheduler.step(train_loss_epoch)

            model.eval()
            val_loss_epoch = 0.
            val_vae_loss_epoch = 0.
            val_regr_loss_epoch = 0.

            with torch.no_grad():
                for x_batch, y_batch in val_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                    regr_output, recon_x, mu, logvar, _ = model(x_batch)
                    vae_loss = model.VAE_loss(recon_x, x_batch, mu, logvar)[0]
                    regr_loss = model.regression_loss(y_batch, regr_output, lambda_tik)

                    val_loss = (1 - beta) * regr_loss + beta * vae_loss

                    val_loss_epoch += val_loss.item()
                    val_vae_loss_epoch += vae_loss.item()
                    val_regr_loss_epoch += regr_loss.item()

            val_loss_epoch /= len(val_loader)
            val_vae_loss_epoch /= len(val_loader)
            val_regr_loss_epoch /= len(val_loader)

            val_regr_loss_per_epoch.append(val_regr_loss_epoch)

            epoch_train_losses[epoch] += train_loss_epoch / num_folds
            epoch_val_losses[epoch] += val_loss_epoch / num_folds
            epoch_val_vae_losses[epoch] += val_vae_loss_epoch / num_folds
            epoch_val_regr_losses[epoch] += val_regr_loss_epoch / num_folds

            print(f"Fold {fold+1} Epoch [{epoch+1}/{num_epochs}] - Total val loss: {epoch_val_losses[epoch]:.4f}, VAE: {epoch_val_vae_losses[epoch]:.4f}, Regr: {epoch_val_regr_losses[epoch]:.4f}")

            if val_regr_loss_epoch < best_val_regr: # early stopping: stop if val regr loss does not imporve for 20 epoxhs
                best_val_regr = val_regr_loss_epoch
                early_stop_counter = 0
                best_model_state = model.state_dict()
            else:
                early_stop_counter += 1

            if early_stop_counter >= patience:
                print(f"Early stopping at epoch {epoch+1} for fold {fold+1}")
                break

        # save best model state for this fold
        fold_model_path = f"4_weights/{sel_zone}/{energies[target_index]}/model_trial_{trial.number}_fold_{fold}.pt"
        torch.save(best_model_state, fold_model_path)
        fold_model_paths.append(fold_model_path)

        # average last 10 epochs val regression loss for fold
        fold_avg_regr_last10 = np.mean(val_regr_loss_per_epoch[-10:])
        fold_last10_regr_losses.append(fold_avg_regr_last10)

    # save paths of all fold models in trial attributes
    trial.set_user_attr("fold_model_paths", fold_model_paths)

    trial.set_user_attr("mean_train_loss_curve", epoch_train_losses.tolist())
    trial.set_user_attr("mean_val_loss_curve", epoch_val_losses.tolist())
    trial.set_user_attr("mean_val_vae_loss_curve", epoch_val_vae_losses.tolist())
    trial.set_user_attr("mean_val_regr_loss_curve", epoch_val_regr_losses.tolist())

    # return average regression loss across folds (last 10 epochs)
    return np.mean(fold_last10_regr_losses)

This cell runs the above study. It can be skipped if the study was already ran and it's only needed to load it.

In [ ]:
# get current day and month
day = date.today().day
month_name = date.today().strftime("%B")

study = optuna.create_study(study_name= f"{energies[target_index]}_study_{day}_{month_name}", direction="minimize",
    storage=f"sqlite:///{energies[target_index]}_optuna.db",
    load_if_exists=True,pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=125))

from tqdm import tqdm
n_trials = 50
study.optimize(objective, n_trials=n_trials,n_jobs=8,catch=(Exception,))

[I 2025-07-01 13:44:48,390] A new study created in RDB with name: wind_study_1_July


Fold 1 Epoch [1/300] - Total val loss: 25.7899, VAE: 894.2562, Regr: 25.3555
Fold 1 Epoch [1/300] - Total val loss: 5.6253, VAE: 815.9345, Regr: 5.2199
Fold 1 Epoch [1/300] - Total val loss: 33.6867, VAE: 879.7083, Regr: 33.2635
Fold 1 Epoch [1/300] - Total val loss: 90.1921, VAE: 817.6538, Regr: 89.8282
Fold 1 Epoch [1/300] - Total val loss: 13.7093, VAE: 814.1014, Regr: 13.3089
Fold 1 Epoch [1/300] - Total val loss: 7.2437, VAE: 813.2348, Regr: 6.8405
Fold 1 Epoch [1/300] - Total val loss: 205.3061, VAE: 813.8135, Regr: 205.0017
Fold 1 Epoch [1/300] - Total val loss: 45.3322, VAE: 816.9169, Regr: 44.9462
Fold 1 Epoch [2/300] - Total val loss: 13.0252, VAE: 964.0588, Regr: 12.5494
Fold 1 Epoch [2/300] - Total val loss: 5.2804, VAE: 811.7458, Regr: 4.8770
Fold 1 Epoch [2/300] - Total val loss: 85.4709, VAE: 816.2489, Regr: 85.1053
Fold 1 Epoch [2/300] - Total val loss: 17.5029, VAE: 943.4475, Regr: 17.0397
Fold 1 Epoch [2/300] - Total val loss: 13.8161, VAE: 824.6260, Regr: 13.4105
Fol

## 3: Loading the best model and plot logic
Assuming the above code has run, a study is available for plotting. Here we want to check the results

In [ ]:
# load the study (mind that the code takes the current day and month: change 'em freely!)
day = '30'#date.today().day
month_name = 'June'#date.today().strftime("%B")

study = optuna.load_study(
    study_name= f"{energies[target_index]}_study_{day}_{month_name}",
    storage=f"sqlite:///{energies[target_index]}_optuna.db"
)

best_trial= study.best_trial
best_params= best_trial.params

best_params

### 3.1: Saving the best model

In [ ]:

# get its best hyperparameters
best_activation = get_activation(best_params["activation"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_paths = best_trial.user_attrs["fold_model_paths"]

best_model = VAE_Regression(
            input_dim= X_train.shape[1],
            hidden_dim1=best_params['hidden_dim1'],
            hidden_dim2=best_params['hidden_dim2'],
            latent_dim=best_params['latent_dim'],
            hidden_dim3=best_params['hidden_dim3'],
            vae_activation= best_activation,
            N1=best_params['N1'],
            N2=best_params['N2'],
            No= Y_test_tensor.shape[1],
            activation= best_activation,
            lambda_tik=best_params['lambda_tik']
        ).to(device)

# load the weights of the best trial
best_model.load_state_dict(torch.load(fold_paths[0], map_location=device))


trial=1
trial += 1
# saves current hyperparamters and weights (which you just loaded) into a new file
with open(f"4_best_models/{sel_zone}/{energies[target_index]}/best_model.json", "w") as f:
    json.dump(best_params, f)

torch.save(best_model.state_dict(), f"best_model_{energies[target_index]}_{day}_{month_name}_trial_{trial}.pt")



### 3.2: Preliminary plots
Plots of this section are not definitive: what we want here is to get an idea of the losses' trends and how the model is performing with a general tuning

In [ ]:
# plot training and validation losses over epochs
z = best_params['latent_dim']
#best_model.load_state_dict(torch.load("lcpb_files/best_model_overall.pt"))

train_loss_curve = np.array(best_trial.user_attrs["mean_train_loss_curve"])
val_loss_curve = np.array(best_trial.user_attrs["mean_val_loss_curve"])
val_vae_loss_curve = np.array(best_trial.user_attrs["mean_val_vae_loss_curve"])
val_regr_loss_curve = np.array(best_trial.user_attrs["mean_val_regr_loss_curve"])

# get only where the loss != 0 (early stopping eventuality)
epochs = np.arange(len(train_loss_curve[train_loss_curve!=0]))

plt.figure(figsize=(12, 6))


plt.rcParams.update({
    'axes.titlesize': 17,
    'axes.labelsize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12.5
})

plt.plot(epochs, train_loss_curve[train_loss_curve!=0], label='Train Loss', color='blue')
plt.plot(epochs, val_loss_curve[val_loss_curve!=0], label='Validation Loss', color='orange')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(fr"Training and Validation Loss over Epochs: $\beta=1/{1./beeta},z={z}$")
plt.legend()
plt.grid(True)
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/3.2_train_val_loss.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
loss_ratio = val_vae_loss_curve / (val_regr_loss_curve + 1e-9)  # avoid division by zero
epochs = np.arange(len(loss_ratio))

plt.figure(figsize=(12, 6))
plt.plot(epochs, loss_ratio, label='VAE / Regression Val loss ratio', color='green')
plt.xlabel("Epoch")
plt.ylabel("Loss Ratio")
plt.title(fr"Ratio of VAE to Regression Validation Loss: $\beta=1/{1./beeta},z={z}$")
plt.grid(True)
plt.legend()
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/3.2_val_loss_ratio.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig,ax= plt.subplots(figsize=(12,6))

ax.plot(epochs, beeta*val_vae_loss_curve, label=r'$\beta \cdot$VAE validation Loss', color='green')
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(fr"Scaled VAE and Regression validation loss curves: $\beta=1/{1./beeta},z={z}$")

ax.plot(epochs[:250], (1-beeta)*val_regr_loss_curve[:250], label=r'$(1-\beta)\cdot$Regression validation Loss', color='red')
plt.ylim(0,25)
ax.grid(True)
ax.legend()
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/3.2_scaled_vae_regr_loss.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')

plt.show()


### 3.3: Model evaluation
The following cells perform a computation of the prediction of the label (this is still preliminary though) and plot the results

In [ ]:
best_model.eval()

num_passes= 200

mean_preds = None

for i in range(num_passes):
    regr_output, recon_x, *_ = best_model(X_test_tensor.to(device))
    regr_output = regr_output.cpu()
    recon_x = recon_x.cpu()

    if mean_preds is None:
        mean_preds = regr_output
        mean_recons = recon_x
    else:
        mean_preds += (regr_output - mean_preds) / (i + 1)
        mean_recons += (recon_x - mean_recons) / (i + 1)

mean_test_predictions = mean_preds.detach().numpy()
mean_recon_x_test = mean_recons.detach().numpy()


print(mean_test_predictions)

In [ ]:
print(np.shape(mean_test_predictions[:,0]))
print(np.shape(Y_test))

In [ ]:
plt.figure(figsize=(13, 8))

# Mask zeros and very small values to avoid extreme divisions
epsilon = 1e-6
mask = np.where(np.abs(Y_test) > epsilon)[0]
masked_Y = Y_test[mask]
masked_pred = mean_test_predictions[mask,0]

# Limit points to plot
points_to_plot = min(200, len(masked_Y))

error_solar = (masked_pred[:points_to_plot] - masked_Y[:points_to_plot]) / masked_Y[:points_to_plot]
error_solar_percent = error_solar * 100

# Remove inf/nan values (just in case)
error_solar_percent = np.nan_to_num(error_solar_percent, nan=0.0, posinf=0.0, neginf=0.0)

plt.plot(np.arange(points_to_plot), error_solar_percent, color='orange',
         label=f'Error of solar production prediction: RMS: {round(np.sqrt(np.mean(error_solar_percent**2)), 1)}%')

plt.xlabel("Index")
plt.ylabel("Prediction error (%)")
plt.title(fr"Percentage error of solar prediction: Test set ($\beta=1/{1./beeta}, z={z}$)")
plt.grid(True)
plt.legend()

# Use lower dpi
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/3.3_label_prediction.png'
plt.savefig(filepath, dpi=150, bbox_inches='tight')
plt.show()

## 4: Search for best $\beta$ parameter
The previous training was all about the structure. We did not run the search over $\beta$, which is the parameter that handles how losses interact, and so determines the effectiveness of our VAE in reducing the dimensionality aiming to a good regression performance. The following cells will try to handle it.

First of all, let's take the previous best parameters and save them. We are going to set these up for the training, this time looking for the best **$\beta$** parameter over a possible range

In [ ]:
hd1 = best_params['hidden_dim1']
hd2 = best_params['hidden_dim2']
z = best_params['latent_dim']
hd3 = best_params['hidden_dim3']
n1 = best_params['N1']
n2 = best_params['N2']
ltk = best_params['lambda_tik']
learning_rate = best_params['lr']
act = best_params['activation']

# beta range and step
beta_min = 10
beta_max = 500
step = 10

### 4.1: Training definition

In [ ]:
# fix previous best hyper parameters and find best beta

def objective(trial):

    def get_activation(name):
        return {
        "relu": nn.ReLU(),
        "sigmoid": nn.Sigmoid(),
        "leaky_relu": nn.LeakyReLU()}[name]

    beta = trial.suggest_float("beta",beta_min,beta_max, step = step) # choose a reasonable range according to previous results
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_epochs = 250
    batch_size = 16
    num_folds = 2
    early_stop_patience = 35
    tscv = TimeSeriesSplit(n_splits= num_folds)

    epoch_train_losses = np.zeros(num_epochs)
    epoch_val_losses = np.zeros(num_epochs)
    epoch_val_vae_losses = np.zeros(num_epochs)
    epoch_val_regr_losses = np.zeros(num_epochs)

    best_val_loss = float('inf')
    best_model_state = None

    fold_val_regr_curves = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_tensor)):

        X_train_fold, Y_train_fold = X_train_tensor[train_idx], Y_train_tensor[train_idx]
        X_val_fold, Y_val_fold = X_train_tensor[val_idx], Y_train_tensor[val_idx]
        
        train_dataset = TensorDataset(X_train_fold, Y_train_fold)
        val_dataset = TensorDataset(X_val_fold, Y_val_fold)
        
        lambda_tik = ltk
        
        model = VAE_Regression(
            input_dim=X_train_fold.shape[1],
            hidden_dim1=hd1,
            hidden_dim2=hd2,
            latent_dim=z,
            hidden_dim3=hd3,
            vae_activation=get_activation('relu'),
            N1=n1,
            N2=n2,
            No=Y_train_fold.shape[1],
            activation=get_activation('relu'),
            lambda_tik=ltk
        ).to(device)

        lr =  learning_rate
        
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience= 20)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        no_improve_counter = 0
        best_fold_loss = float('inf')
        fold_val_regr_curve = []

        for epoch in range(num_epochs):

            model.train()
            train_loss_epoch = 0
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()

                regr_output, recon_x, mu, logvar, _ = model(x_batch)
                vae_loss = model.VAE_loss(recon_x, x_batch, mu, logvar)[0]
                regr_loss = model.regression_loss(y_batch, regr_output, lambda_tik)

                loss = (1-1./beta) * regr_loss + 1./beta*vae_loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss_epoch += loss.item()

            train_loss_epoch /= len(train_loader)
            scheduler.step(train_loss_epoch)

            model.eval()
            val_loss_epoch = 0.
            val_vae_loss_epoch = 0.
            val_regr_loss_epoch = 0.

            with torch.no_grad():
                for x_batch, y_batch in val_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                    regr_output, recon_x, mu, logvar, _ = model(x_batch)
                    vae_loss = model.VAE_loss(recon_x, x_batch, mu, logvar)[0]
                    regr_loss = model.regression_loss(y_batch, regr_output, lambda_tik)

                    val_loss = (1 - 1. / beta) * regr_loss + 1. / beta * vae_loss

                    val_loss_epoch += val_loss.item()
                    val_vae_loss_epoch += vae_loss.item()
                    val_regr_loss_epoch += regr_loss.item()

            val_loss_epoch /= len(val_loader)
            val_vae_loss_epoch /= len(val_loader)
            val_regr_loss_epoch /= len(val_loader)

            epoch_train_losses[epoch] += (train_loss_epoch) / num_folds
            epoch_val_losses[epoch] += (val_loss_epoch) / num_folds
            epoch_val_vae_losses[epoch] += (val_vae_loss_epoch) / num_folds
            epoch_val_regr_losses[epoch] += (val_regr_loss_epoch) / num_folds

            fold_val_regr_curve.append(val_regr_loss_epoch)

            print(f"Epoch [{epoch + 1}/{num_epochs}] - Total val loss: {epoch_val_losses[epoch]:.4f}, VAE: {epoch_val_vae_losses[epoch]:.4f}, Regr: {epoch_val_regr_losses[epoch]:.4f}")

            if val_loss_epoch < best_fold_loss: # early stopping
                best_fold_loss = val_loss_epoch
                no_improve_counter = 0
            else:
                no_improve_counter += 1
                if no_improve_counter >= early_stop_patience:
                    print(f"Early stopping at epoch {epoch + 1} in fold {fold + 1}")
                    break

            if val_loss_epoch < best_val_loss:
                best_val_loss = val_loss_epoch
                best_model_state = model.state_dict()

        fold_val_regr_curves.append(fold_val_regr_curve)

    fold_model_path = f"4_weights/{sel_zone}/{energies[target_index]}/beta_trial_{trial.number}_fold_{fold}.pt"
    torch.save(best_model_state, fold_model_path)
    trial.set_user_attr("model_path", fold_model_path)

    trial.set_user_attr("mean_train_loss_curve", epoch_train_losses.tolist())
    trial.set_user_attr("mean_val_loss_curve", epoch_val_losses.tolist())
    trial.set_user_attr("mean_val_vae_loss_curve", epoch_val_vae_losses.tolist())
    trial.set_user_attr("mean_val_regr_loss_curve", epoch_val_regr_losses.tolist())

    trial.set_user_attr("final_vae_loss", epoch_val_vae_losses[-1])
    trial.set_user_attr("final_regr_loss", epoch_val_regr_losses[-1])

    mean_last10_regr_losses = []
    for curve in fold_val_regr_curves:
        if len(curve) >= 10:
            mean_last10_regr_losses.append(np.mean(curve[-10:]))
        else:
            mean_last10_regr_losses.append(np.mean(curve))

    return float(np.mean(mean_last10_regr_losses))


The following cell runs the training (if it was not ran before). If there is already a study in memory, this cell can be skipped

In [ ]:
from tqdm import tqdm
n_trials = 30

# get current day and month
day = date.today().day
month_name = date.today().strftime("%B")


max_jobs = min(n_trials, os.cpu_count() or 4)  # for SQLite, avoid too many writers

study_beta = optuna.create_study( study_name= f"{energies[target_index]}_study_beta_{day}_{month_name}", direction="minimize",
    storage=f"sqlite:///{energies[target_index]}_optuna.db",
    load_if_exists=True,pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=125))


study_beta.optimize(objective, n_trials=n_trials,n_jobs=max_jobs,catch=(Exception,))

### 4.2: Retrieving the best model and final results

In [ ]:
best_trial = study_beta.best_trial
best_params = best_trial.params
best_params

In [ ]:
# final training for the best hyper-parameters found

import os
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset


best_trial= study_beta.best_trial
best_params= best_trial.params
beta = 1./best_params['beta']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lambda_tik= ltk
best_model = VAE_Regression(
     input_dim=X_train_tensor.shape[1],
     hidden_dim1=hd1,
     hidden_dim2=hd2,
     latent_dim=z,
     hidden_dim3=hd3,
     vae_activation=get_activation('relu'),
     N1=n1,
     N2=n2,
     No=Y_train_tensor.shape[1],
     activation=get_activation('relu'),
     lambda_tik=lambda_tik).to(device)

# REMOVE NAN
# Get mask for rows that are valid (no NaNs in any Y_test_tensor column)
valid_mask = ~torch.isnan(Y_test_tensor).any(dim=1)

# Apply mask to both X and Y tensors
X_test_tensor = X_test_tensor[valid_mask]
Y_test_tensor = Y_test_tensor[valid_mask]


train_loader = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor), batch_size=16, shuffle=True, drop_last = True)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  Y_test_tensor),  batch_size=16, shuffle=False, drop_last = False)

num_epochs = 250
#beta       = 1/550.0 # best previously found
patience   = 35

train_curve     = []
test_curve      = []
test_vae_curve  = []
test_regr_curve = []

optimizer = torch.optim.Adam(best_model.parameters(), lr=0.00011695038421675327)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=20)

best_test = float("inf")
no_imp    = 0
best_state= None

for ep in range(num_epochs):
    best_model.train()
    train_sum = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred, recon, mu, logvar, _ = best_model(xb)
        vl = best_model.VAE_loss(recon, xb, mu, logvar)[0]
        rl = best_model.regression_loss(yb, pred, lambda_tik)
        loss = (1-beta)*rl + beta*vl
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
        optimizer.step()
        train_sum += loss.item()
    train_curve.append(train_sum/len(train_loader))

    best_model.eval()
    t_sum = v_sum = r_sum = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred, recon, mu, logvar, _ = best_model(xb)
            vl = best_model.VAE_loss(recon, xb, mu, logvar)[0].item()
            rl = best_model.regression_loss(yb, pred,lambda_tik).item()
            t_sum += (1-beta)*rl + beta*vl
            v_sum += vl
            r_sum += rl
    te = t_sum/len(test_loader)
    tv = v_sum/len(test_loader)
    tr = r_sum/len(test_loader)
    test_curve.append(te)
    test_vae_curve.append(tv)
    test_regr_curve.append(tr)

    print(f"Epoch {ep+1}/{num_epochs} - Val Total: {te:.4f}, VAE: {tv:.4f}, Regr: {tr:.4f}")

    
    if tr < best_test:
        best_test = tr
        best_state= best_model.state_dict()
        no_imp    = 0
    else:
        no_imp += 1
        if no_imp >= patience:
            break

best_model.load_state_dict(best_state)
os.makedirs("best_Regr_models/Single_solar", exist_ok=True)
torch.save(best_state, "best_Regr_models/Single_solar/best_model_solar_23June_FINAL.pt")

train_loss_curve    = np.array(train_curve)
val_loss_curve      = np.array(test_curve)
val_vae_loss_curve  = np.array(test_vae_curve)
val_regr_loss_curve = np.array(test_regr_curve)


# 5: Visualization of reconstructed features and final results
### 5.1: final results and losses

In [ ]:
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/5.1_pred_vs_test_final.png'

best_model.eval()
num_passes = 100

mean_preds = None
for i in range(num_passes):
    regr_output, recon_x, *_ = best_model(X_test_tensor.to(device))
    regr_output = regr_output.cpu()

    if mean_preds is None:
        mean_preds = regr_output
        mean_recons = recon_x
    else:
        mean_preds += (regr_output - mean_preds) / (i + 1)
        mean_recons += (recon_x - mean_recons) / (i + 1)

mean_test_predictions = mean_preds.detach().numpy()
mean_recon_x_test = mean_recons.detach().numpy()

mean_test_predictions

In [ ]:
plt.rcParams.update({
    'axes.titlesize': 17,
    'axes.labelsize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12.5
    })

points_to_plot = 220
energy_colors = ['red','green','blue']

plt.figure(figsize = (14,10))
plt.plot(np.arange(points_to_plot),mean_test_predictions[:points_to_plot], color = energy_colors[target_index], label = f'{energies[target_index]} production prediction')
plt.plot(np.arange(points_to_plot),Y_test[:points_to_plot],color ='orange', label = 'Test data')
plt.xlabel("Index")
plt.ylabel(f"Normalized {energies[target_index]} energy production")
plt.title(fr"Prediction of {energies[target_index]} energy production: Test set ($\beta=1/{best_params['beta']}, z={z}$)")
plt.grid(True)
plt.legend()
plt.savefig(filepath, dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
# plot training and validation losses over epochs
#best_model.load_state_dict(torch.load("lcpb_files/best_model_overall.pt"))

train_loss_curve = np.array(best_trial.user_attrs["mean_train_loss_curve"])
val_loss_curve = np.array(best_trial.user_attrs["mean_val_loss_curve"])
val_vae_loss_curve = np.array(best_trial.user_attrs["mean_val_vae_loss_curve"])
val_regr_loss_curve = np.array(best_trial.user_attrs["mean_val_regr_loss_curve"])

# get only where the loss != 0 (early stopping eventuality)
epochs = np.arange(len(train_loss_curve[train_loss_curve!=0]))

plt.figure(figsize=(12, 6))


plt.rcParams.update({
    'axes.titlesize': 17,
    'axes.labelsize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12.5
})

plt.plot(epochs, train_loss_curve[train_loss_curve!=0], label='Train Loss', color='blue')
plt.plot(epochs, val_loss_curve[val_loss_curve!=0], label='Validation Loss', color='orange')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(fr"Training and Validation Loss over Epochs: $\beta=1/{1./beeta},z={z}$")
plt.legend()
plt.grid(True)
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/5.1_train_val_loss.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
loss_ratio = val_vae_loss_curve / (val_regr_loss_curve + 1e-9)  # avoid division by zero
loss_ratio = loss_ratio[loss_ratio > 0]
epochs = np.arange(len(loss_ratio))

plt.figure(figsize=(12, 6))
plt.plot(epochs, loss_ratio, label='VAE / Regression Val loss ratio', color='green')
plt.xlabel("Epoch")
plt.ylabel("Loss Ratio")
plt.title(fr"Ratio of VAE to Regression Validation Loss: $\beta=1/{1./beeta},z={z}$")
plt.grid(True)
plt.legend()
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/5.1_val_loss_ratio.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig,ax= plt.subplots(figsize=(12,6))
epochs = np.arange(250)
ax.plot(epochs, beta*val_vae_loss_curve, label=r'$\beta \cdot$VAE validation Loss', color='green')
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(fr"Scaled VAE and Regression validation loss curves: $\beta=1/{1./beeta},z={z}$")

ax.plot(epochs, (1-beta)*val_regr_loss_curve[:250], label=r'$(1-\beta)\cdot$Regression validation Loss', color='red')
plt.ylim(0,10)
ax.grid(True)
ax.legend()
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/5.1_scaled_vae_regr_loss.png'
plt.savefig(filepath, dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
plt.figure(figsize=(13, 8))

# Mask zeros and very small values to avoid extreme divisions
epsilon = 1e-6
mask = np.where(np.abs(Y_test) > epsilon)[0]
masked_Y = Y_test[mask]
masked_pred = mean_test_predictions[mask,0]

# Limit points to plot
points_to_plot = min(200, len(masked_Y))

error_solar = (masked_pred[:points_to_plot] - masked_Y[:points_to_plot]) / masked_Y[:points_to_plot]
error_solar_percent = error_solar * 100

# Remove inf/nan values (just in case)
error_solar_percent = np.nan_to_num(error_solar_percent, nan=0.0, posinf=0.0, neginf=0.0)

plt.plot(np.arange(points_to_plot), error_solar_percent, color='orange',
         label=f'Error of solar production prediction: RMS: {round(np.sqrt(np.mean(error_solar_percent**2)), 1)}%')

plt.xlabel("Index")
plt.ylabel("Prediction error (%)")
plt.title(fr"Percentage error of solar prediction: Test set ($\beta=1/{1./beeta}, z={z}$)")
plt.grid(True)
plt.legend()

# Use lower dpi
filepath = f'4_plots/{sel_zone}/{energies[target_index]}/5.1_error_prediction.png'
plt.savefig(filepath, dpi=150, bbox_inches='tight')
plt.show()

### 5.2: reconstructing features

In [ ]:
original_all = X_test
reconstructed_all = mean_recon_x_test

feature_names = df_features.drop(columns = ['timestamp']).columns.tolist()
features_to_plot = np.arange(0,10,1)

points_to_plot = 200
dummy_x = np.arange(points_to_plot)

for feature_idx in features_to_plot:
    plt.figure(figsize = (8,6))
    plt.plot(dummy_x,
            original_all[:points_to_plot,feature_idx],
            alpha = 0.8,
            label = f'Original {feature_names[feature_idx]}')
    plt.plot(dummy_x,
            reconstructed_all[:points_to_plot,feature_idx],
            alpha = 0.8,
            label = f'Reconstructed {feature_names[feature_idx]}')
    plt.xlabel('Point index')
    plt.ylabel('Feature value')
    plt.title(f'Feature {feature_names[feature_idx]} reconstruction')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    filename = f'4_plots/{sel_zone}/{energies[target_index]}/5.2_VAE_reconstruction_feature_{feature_idx}.png'
    plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()